## Execução de Limpeza: PEDE 2022
Com base no diagnóstico da EDA, aplicamos agora as transformações necessárias para padronizar o dataset de 2022. 
As ações incluem:
1. **Padronização de Gênero** para Masculino/Feminino.
2. **Clusterização de Instituições** em Pública/Privada.
3. **Imputação de Notas Nulas** (Zero-fill).
4. **Normalização de Texto da Fase Ideal** (Remoção de parênteses e espaços duplos) e criação de nova coluna

In [155]:
import pandas as pd

# O parâmetro sheet_name=None diz ao Pandas para carregar TUDO
todas_as_folhas = pd.read_excel('data/base_dados_2024.xlsx', sheet_name=None)

# Para ver os nomes de todas as folhas que foram carregadas:
print(todas_as_folhas.keys())


# Agora, pode separar cada folha no seu próprio DataFrame usando o nome exato da folha:
df_2022 = todas_as_folhas['PEDE2022']

dict_keys(['PEDE2022', 'PEDE2023', 'PEDE2024'])


# Ação 1: Padronização do Gênero 
Os outros datasets estão padronizados como Masculino e Feminino

In [156]:
import numpy as np

df_2022['Gênero'] = df_2022['Gênero'].replace({
    'Menino': 'Masculino', 
    'Menina': 'Feminino'
})
print("--- Verificação de Gênero ---")
print(df_2022['Gênero'].value_counts(dropna=False))

--- Verificação de Gênero ---
Gênero
Feminino     457
Masculino    403
Name: count, dtype: int64


# Ação 2: Agrupamento de Instituição de Ensino

In [157]:
# Ação 2: Agrupamento de Instituição de Ensino
df_2022['Instituição de ensino'] = df_2022['Instituição de ensino'].replace({
    'Escola Pública': 'Pública',
    'Rede Decisão': 'Privada',
    'Escola JP II': 'Privada'
})
print("--- Verificação de Instituições (Pós-Agrupamento) ---")
# O dropna=False serve para checar se algum valor nulo apareceu
print(df_2022['Instituição de ensino'].value_counts(dropna=False))

--- Verificação de Instituições (Pós-Agrupamento) ---
Instituição de ensino
Pública    752
Privada    108
Name: count, dtype: int64


# Ação 3: corrigindo via regra de negócio. 
Imputação por Similaridade: Identificamos dois alunos (RA-288 e RA-294) que possuíam um INDE alto, mas estavam sem as notas de Matemática e Português. Em vez de preencher com zero (o que seria um erro estatístico), calculamos a média de seus "pares" (alunos da mesma Fase e com performance similar) para manter a coerência da fórmula do INDE.

In [158]:
#  Identificamos o IDA dos alunos com problema 
# Se o IDA não existir, usamos o INDE como proxy 
coluna_referencia = 'IDA 22' if 'IDA 22' in df_2022.columns else 'INDE 22'

# Calculamos a média de Mat/Por de alunos que estão no mesmo "nível" de IDA/INDE
# Filtramos alunos da mesma Fase (3) com performance similar (entre 4 e 6)
peers_equivalentes = df_2022[(df_2022['Fase'] == 3) & (df_2022[coluna_referencia].between(4, 6))]

media_mat = round(peers_equivalentes['Matem'].mean(), 2)
media_por = round(peers_equivalentes['Portug'].mean(), 2)

print(f"Usando {coluna_referencia} como base para o cálculo.")
print(f"Média Sugerida -> Mat: {media_mat} | Por: {media_por}")

# Aplicamos a correção cirúrgica
RAs_problema = ['RA-288', 'RA-294']
df_2022.loc[df_2022['RA'].isin(RAs_problema), 'Matem'] = media_mat
df_2022.loc[df_2022['RA'].isin(RAs_problema), 'Portug'] = media_por

print(f"Notas imputadas com sucesso para {RAs_problema}")

Usando INDE 22 como base para o cálculo.
Média Sugerida -> Mat: 3.45 | Por: 4.14
Notas imputadas com sucesso para ['RA-288', 'RA-294']


In [159]:
# Mostra apenas as linhas onde pelo menos uma das notas é NaN
cols_analise = ['Matem', 'Portug', 'Inglês', 'INDE 22']
nulos_restantes = df_2022[df_2022[cols_analise].isnull().any(axis=1)]

print(f"\nTotal de alunos com alguma nota faltando: {len(nulos_restantes)}")
# Exibimos o RA e a Fase para entender o contexto (ex: se são todos da Fase ALFA)
print(nulos_restantes[['RA', 'Fase', 'Matem', 'Portug', 'Inglês', 'INDE 22']])


Total de alunos com alguma nota faltando: 577
         RA  Fase  Matem  Portug  Inglês  INDE 22
13    RA-14     7    8.0     0.0     NaN    6.793
14    RA-15     7    8.0     6.5     NaN    7.783
15    RA-16     7    7.0     0.0     NaN    6.112
16    RA-17     7    9.0     6.5     NaN    8.068
17    RA-18     7    9.0     9.0     NaN    7.226
..      ...   ...    ...     ...     ...      ...
855  RA-856     0    8.5     8.2     NaN    8.398
856  RA-857     0    9.6     9.3     NaN    8.154
857  RA-858     0    8.4     6.9     NaN    7.523
858  RA-859     0    9.4     8.4     NaN    8.325
859  RA-860     0    8.9     7.2     NaN    8.136

[577 rows x 6 columns]


In [160]:
print("\nQuem são eles?")
# Agrupa por fase e conta quantos nulos existem em cada uma
print(df_2022.groupby('Fase')[cols_analise].apply(lambda x: x.isnull().sum()))


Quem são eles?
      Matem  Portug  Inglês  INDE 22
Fase                                
0         0       0     190        0
1         0       0     192        0
2         0       0     155        0
3         0       0      21        0
4         0       0       3        0
5         0       0       8        0
6         0       0       0        0
7         0       0       8        0


 Precisamos garantir que o modelo no futuro entenda quem tem inglês e não tem, se imputar 0 o modelo vai entender que essas pessoas foram mal na prova de inglês

In [161]:
# Criamos uma coluna que diz se o aluno REALMENTE tem a matéria
df_2022['Tem_Ingles'] = df_2022['Inglês'].apply(lambda x: 0 if pd.isna(x) else 1)

# Agora sim, preenchemos o original com 0 para não quebrar os cálculos
df_2022['Inglês'] = df_2022['Inglês'].fillna(0)

In [162]:
print(df_2022.groupby('Fase')[cols_analise].apply(lambda x: x.isnull().sum()))

      Matem  Portug  Inglês  INDE 22
Fase                                
0         0       0       0        0
1         0       0       0        0
2         0       0       0        0
3         0       0       0        0
4         0       0       0        0
5         0       0       0        0
6         0       0       0        0
7         0       0       0        0


# Ação 4 - Limpeza da 'Fase ideal'
 1. Usamos o split('(') para cortar tudo o que vem depois do parênteses
 2. Usamos o strip() para remover espaços extras (incluindo o espaço duplo do ALFA)

In [163]:
# Desmembro a coluna 'Fase ideal' em duas
# O expand=True já cria as duas colunas de uma vez
df_2022[['Fase', 'Serie_Escolar']] = df_2022['Fase ideal'].str.split(r' \(', expand=True)

# Limpeza do parênteses que sobra na direita e espaços extras
df_2022['Serie_Escolar'] = df_2022['Serie_Escolar'].str.replace(')', '', regex=False).str.strip()
df_2022['Fase'] = df_2022['Fase'].str.strip()

# Ajuste do ALFA (Tratando o espaço duplo e a série vazia)
df_2022.loc[df_2022['Fase'].str.contains('ALFA', na=False), 'Fase'] = 'ALFA'
df_2022.loc[df_2022['Fase'] == 'ALFA', 'Serie_Escolar'] = 'Alfabetização'

# --- CHECK-UP RÁPIDO ---
print(df_2022[['Fase', 'Serie_Escolar']].value_counts())

Fase    Serie_Escolar 
Fase 2  5º e 6º ano       218
Fase 3  7º e 8º ano       207
Fase 1  4º ano             96
Fase 4  9º ano             85
ALFA    Alfabetização      71
Fase 5  1º EM              63
Fase 6  2º EM              52
Fase 7  3º EM              48
Fase 8  Universitários     20
Name: count, dtype: int64


In [164]:
# Só para garantir que não restou nenhum espaço invisível nas novas categorias
df_2022['Serie_Escolar'] = df_2022['Serie_Escolar'].str.strip()

print("Série Escolar higienizada!")

Série Escolar higienizada!


# Investigação INDE

In [165]:
# --- AUDITORIA DE PESOS REFINADA ---
print("--- INTERROGANDO OS PESOS DE 2022 ---")

# Criamos uma cópia temporária preenchendo nulos com 0 para a conta bater
df_temp = df_2022.fillna(0)

# Base fixa (pesos que mantiveram 0.7 do total)
soma_fixa = (
    df_temp['IDA'] * 0.2 + df_temp['IEG'] * 0.2 + 
    df_temp['IAA'] * 0.1 + df_temp['IPV'] * 0.2
)

# Hipótese A: IAN subiu para 0.2 (IPS continua 0.1)
h_ian = abs(df_temp['INDE 22'] - (soma_fixa + df_temp['IAN'] * 0.2 + df_temp['IPS'] * 0.1))

# Hipótese B: IPS subiu para 0.2 (IAN continua 0.1)
h_ips = abs(df_temp['INDE 22'] - (soma_fixa + df_temp['IAN'] * 0.1 + df_temp['IPS'] * 0.2))

# Hipótese C: IAA subiu para 0.2 (IAN e IPS continuam 0.1) - Reajustando a soma_fixa
soma_fixa_c = (df_temp['IDA'] * 0.2 + df_temp['IEG'] * 0.2 + df_temp['IPV'] * 0.2)
h_iaa = abs(df_temp['INDE 22'] - (soma_fixa_c + df_temp['IAA'] * 0.2 + df_temp['IAN'] * 0.1 + df_temp['IPS'] * 0.1))

# Resultados (olhamos o erro médio)
print(f"Erro Médio Hipótese A (IAN 0.2): {h_ian.mean():.4f}")
print(f"Erro Médio Hipótese B (IPS 0.2): {h_ips.mean():.4f}")
print(f"Erro Médio Hipótese C (IAA 0.2): {h_iaa.mean():.4f}")

if h_ian.mean() < 0.01:
    print("\n CONCLUSÃO: O peso do IPP foi para o IAN!")
elif h_ips.mean() < 0.01:
    print("\n CONCLUSÃO: O peso do IPP foi para o IPS!")
elif h_iaa.mean() < 0.01:
    print("\n CONCLUSÃO: O peso do IPP foi para o IAA!")
else:
    print("\n Nenhuma hipótese padrão bateu. A ONG pode ter distribuído os 0.1 entre todos os indicadores.")

--- INTERROGANDO OS PESOS DE 2022 ---
Erro Médio Hipótese A (IAN 0.2): 0.2054
Erro Médio Hipótese B (IPS 0.2): 0.1384
Erro Médio Hipótese C (IAA 0.2): 0.2697

 Nenhuma hipótese padrão bateu. A ONG pode ter distribuído os 0.1 entre todos os indicadores.


In [166]:
colunas_remover = [
    'INDE 23', 'Pedra 23', 'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4', 'Rec Psicologia', 
    'Destaque IEG', 'Destaque IDA', 'Destaque IPV', 'Destaque IPV.1',
    'Indicado', 'Atingiu PV', 'Avaliador3', 'Avaliador4', 'Avaliador1' , 'Avaliador2' , 'Nº Av' 
]
df_2022.drop(columns=colunas_remover, inplace=True, errors='ignore')

In [167]:
# =====================================================================
# AUDITORIA FINAL DE VALORES NULOS
# =====================================================================
print("\n" + "="*50)
print("--- VERIFICAÇÃO DE DADOS FALTANTES ---")

# Conta os nulos por coluna
nulos_por_coluna = df_2022.isnull().sum()

# Filtra apenas as colunas que têm mais de 0 nulos e ordena da maior para a menor
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna >= 0].sort_values(ascending=False)

# Verifica se o filtro encontrou alguma coisa
if colunas_com_nulos.empty:
    print(" SUCESSO! O dataset de 2023 está 100% limpo, sem nenhum valor nulo.")
else:
    print("ATENÇÃO! As seguintes colunas ainda possuem valores nulos:\n")
    
    # Monta uma tabela formatada para exibir o Nome da Coluna, a Quantidade e a %
    tabela_nulos = pd.DataFrame({
        'Quantidade de Nulos': colunas_com_nulos,
        'Porcentagem (%)': (colunas_com_nulos / len(df_2022)) * 100
    })
    
    # Exibe a tabela com 2 casas decimais
    print(tabela_nulos.round(2))

print("="*50 + "\n")


--- VERIFICAÇÃO DE DADOS FALTANTES ---
ATENÇÃO! As seguintes colunas ainda possuem valores nulos:

                       Quantidade de Nulos  Porcentagem (%)
Pedra 20                               537            62.44
Pedra 21                               398            46.28
Turma                                    0             0.00
Fase                                     0             0.00
RA                                       0             0.00
Ano nasc                                 0             0.00
Nome                                     0             0.00
Idade 22                                 0             0.00
Gênero                                   0             0.00
Instituição de ensino                    0             0.00
Ano ingresso                             0             0.00
Pedra 22                                 0             0.00
INDE 22                                  0             0.00
Cg                                       0             0.00


# Renomeação das colunas

In [168]:
#Dicionário de Mapeamento (De -> Para)
map_22 = {
    'RA': 'ra', 'Nome': 'nome', 'Gênero': 'genero', 'Idade 22': 'idade',
    'Ano ingresso': 'ano_ingresso', 'Fase': 'fase', 'Turma': 'turma',
    'Atingiu PV': 'ponto_virada', 
    'INDE 22': 'inde',     
    'Pedra 22': 'pedra',   
    'Pedra 21': 'pedra_2021',
    'Pedra 20': 'pedra_2020',
    'Matem': 'mat', 'Portug': 'por', 'Inglês': 'ing',
    'Instituição de ensino': 'Instituicao de ensino'
}

#  Aplicar o rename
df_2022 = df_2022.rename(columns=map_22)

# 2. CRIAR A COLUNA DE REFERÊNCIA (Fundamental para o padrão Long)
df_2022['ano_referencia'] = 2022

#  Limpeza residual: colocar tudo em minúsculo e trocar espaços por underline
df_2022.columns = [col.lower().replace(' ', '_') for col in df_2022.columns]

# NOVO SHAPE 
print(df_2022.columns.tolist())

# Verificação rápida da nova coluna
print(f"\nExemplo de registro com ano: {df_2022['ano_referencia'].iloc[0]}")

# 4. Salvando a versão DEFINITIVA
df_2022.to_csv('pede_2022_final.csv', index=False, encoding='utf-8-sig')
print("\nBase 2022 Padronizada, com Ano de Referência e Salva!")

['ra', 'fase', 'turma', 'nome', 'ano_nasc', 'idade', 'genero', 'ano_ingresso', 'instituicao_de_ensino', 'pedra_2020', 'pedra_2021', 'pedra', 'inde', 'cg', 'cf', 'ct', 'iaa', 'ieg', 'ips', 'ida', 'mat', 'por', 'ing', 'ipv', 'ian', 'fase_ideal', 'defas', 'tem_ingles', 'serie_escolar', 'ano_referencia']

Exemplo de registro com ano: 2022

Base 2022 Padronizada, com Ano de Referência e Salva!
